# OpportunityMatch AI — Mini Semantic Search Engine

### AWS AI/ML Track — Week 02 Assignment

**Project Goal:** Build a working semantic search engine that helps students discover relevant hackathons, internships, scholarships, workshops, bootcamps, fellowships, and competitions using natural-language queries. The system finds opportunities based on contextual meaning rather than just exact keyword matches.

---

## 1. Setup and Installation
We will install `sentence-transformers` for generating vector embeddings and `chromadb` as our lightweight vector database.

In [23]:
!pip install -q sentence-transformers chromadb pandas numpy

## 2. Imports and Environment Setup
Let's import all the necessary standard libraries and verify our environment.

In [24]:
import os
import pandas as pd
import numpy as np
import chromadb
from sentence_transformers import SentenceTransformer

print("Libraries successfully imported!")

Libraries successfully imported!


## 3. Creating a Self-Contained Sample Dataset
We will create a robust local CSV dataset with 20 diverse opportunities to ensure this notebook remains completely self-contained and reproducible without manual file uploads.

In [25]:
import csv

opportunities = [
    {
        "id": "OPP_001",
        "title": "Global AI Hackathon",
        "description": "A weekend hackathon focused on building generative AI applications and machine learning models to solve real-world climate issues.",
        "category": "Hackathon",
        "skills": "Python, PyTorch, Generative AI, Machine Learning",
        "location": "Virtual",
        "eligibility": "All university students globally",
        "deadline": "2025-05-15 (Demonstration Only)",
        "url": "https://example.com/global-ai-hackathon"
    },
    {
        "id": "OPP_002",
        "title": "Data Science Internship - NextGen Analytics",
        "description": "Summer internship program for aspiring data scientists. Work with experienced mentors on predictive analytics, data cleaning, and business intelligence dashboards.",
        "category": "Internship",
        "skills": "SQL, Python, Pandas, Tableau",
        "location": "Chicago, IL / Hybrid",
        "eligibility": "Rising juniors and seniors majoring in STEM fields",
        "deadline": "2025-04-01 (Demonstration Only)",
        "url": "https://example.com/nextgen-internship"
    },
    {
        "id": "OPP_003",
        "title": "Women in Tech Scholarship",
        "description": "Financial aid and mentorship program dedicated to supporting female-identifying undergraduate students pursuing computer science or software engineering degrees.",
        "category": "Scholarship",
        "skills": "Computer Science foundation, leadership potential",
        "location": "Global",
        "eligibility": "Female undergraduate computer science majors",
        "deadline": "2025-06-30 (Demonstration Only)",
        "url": "https://example.com/women-in-tech-scholarship"
    },
    {
        "id": "OPP_004",
        "title": "Cloud Computing and AWS Workshop",
        "description": "A hands-on, 1-day virtual workshop guiding beginners through AWS basics, EC2 instances, S3 storage, and serverless computing.",
        "category": "Workshop",
        "skills": "AWS, Cloud Architecture, Basic Linux commands",
        "location": "Virtual",
        "eligibility": "Open to all beginners",
        "deadline": "2025-03-10 (Demonstration Only)",
        "url": "https://example.com/aws-workshop"
    },
    {
        "id": "OPP_005",
        "title": "Full-Stack Web Development Bootcamp",
        "description": "An intensive 12-week coding program designed to take you from a complete beginner to building full-stack web applications with modern frameworks.",
        "category": "Bootcamp",
        "skills": "HTML, CSS, JavaScript, React, Node.js",
        "location": "Boston, MA / In-person",
        "eligibility": "Anyone interested in switching careers or learning web dev",
        "deadline": "2025-04-15 (Demonstration Only)",
        "url": "https://example.com/web-dev-bootcamp"
    },
    {
        "id": "OPP_006",
        "title": "Graduate Fellowship in Quantum Computing",
        "description": "Prestigious fellowship providing research funding and lab access for PhD students investigating quantum algorithms and hardware.",
        "category": "Fellowship",
        "skills": "Quantum Mechanics, Advanced Physics, Qiskit",
        "location": "Palo Alto, CA",
        "eligibility": "First or second-year PhD students",
        "deadline": "2025-02-28 (Demonstration Only)",
        "url": "https://example.com/quantum-fellowship"
    },
    {
        "id": "OPP_007",
        "title": "Cyber Security CTF (Capture the Flag)",
        "description": "An exciting competitive event testing your skills in cryptography, reverse engineering, web exploitation, and digital forensics.",
        "category": "Competition",
        "skills": "Ethical Hacking, Linux, Cryptography, Wireshark",
        "location": "Virtual",
        "eligibility": "Individual or team registrations for high school and college students",
        "deadline": "2025-05-01 (Demonstration Only)",
        "url": "https://example.com/ctf-competition"
    },
    {
        "id": "OPP_008",
        "title": "Mobile App Innovation Challenge",
        "description": "Form teams of up to 4 to design and develop an innovative mobile application using Flutter or React Native that addresses a community health need.",
        "category": "Competition",
        "skills": "Mobile Development, UI/UX Design, Flutter, React Native",
        "location": "Virtual",
        "eligibility": "High school and university teams",
        "deadline": "2025-07-20 (Demonstration Only)",
        "url": "https://example.com/mobile-app-challenge"
    },
    {
        "id": "OPP_009",
        "title": "AI Product Management Fellowship",
        "description": "A 6-month part-time program pairing students with product managers at leading tech companies to learn how to design and ship AI-powered products.",
        "category": "Fellowship",
        "skills": "Product Strategy, Product Design, AI/ML concepts",
        "location": "Remote",
        "eligibility": "Undergraduate and graduate students",
        "deadline": "2025-04-10 (Demonstration Only)",
        "url": "https://example.com/pm-fellowship"
    },
    {
        "id": "OPP_010",
        "title": "Software Engineer Intern - Cloud Systems",
        "description": "Join our infrastructure team to build highly scalable backend microservices, write unit tests, and maintain CI/CD pipelines on AWS.",
        "category": "Internship",
        "skills": "Go, Python, Docker, Kubernetes, AWS",
        "location": "Seattle, WA / Hybrid",
        "eligibility": "Computer Science or Software Engineering majors",
        "deadline": "2025-03-15 (Demonstration Only)",
        "url": "https://example.com/cloud-internship"
    },
    {
        "id": "OPP_011",
        "title": "Introduction to Cybersecurity Bootcamp",
        "description": "A comprehensive entry-level training bootcamp covers network defense, risk mitigation, and security compliance basics.",
        "category": "Bootcamp",
        "skills": "Network Security, CompTIA Security+, Firewalls",
        "location": "Virtual",
        "eligibility": "No prior technical experience required",
        "deadline": "2025-06-01 (Demonstration Only)",
        "url": "https://example.com/cyber-bootcamp"
    },
    {
        "id": "OPP_012",
        "title": "STEM Excellence Scholarship",
        "description": "Merit-based financial award for underprivileged students entering accredited bachelor's programs in STEM.",
        "category": "Scholarship",
        "skills": "Mathematics, Science, Academic Achievement",
        "location": "United States",
        "eligibility": "Incoming college freshmen from low-income households",
        "deadline": "2025-05-30 (Demonstration Only)",
        "url": "https://example.com/stem-scholarship"
    },
    {
        "id": "OPP_013",
        "title": "Git and GitHub Collaboration Seminar",
        "description": "Short hands-on training focusing on version control, branching, pull requests, and resolving merge conflicts in collaborative coding projects.",
        "category": "Workshop",
        "skills": "Git, GitHub, Collaboration",
        "location": "Virtual",
        "eligibility": "Open to all tech enthusiasts",
        "deadline": "2025-03-25 (Demonstration Only)",
        "url": "https://example.com/git-seminar"
    },
    {
        "id": "OPP_014",
        "title": "UX/UI Design Challenge",
        "description": "Design a modern desktop portal that simplifies student resource discovery. Submissions are judged on usability, user research, and interactive prototyping.",
        "category": "Competition",
        "skills": "Figma, Prototyping, User Research, Wireframing",
        "location": "Virtual",
        "eligibility": "Individuals or teams of 2",
        "deadline": "2025-08-05 (Demonstration Only)",
        "url": "https://example.com/ux-design-challenge"
    },
    {
        "id": "OPP_015",
        "title": "Machine Learning Research Internship",
        "description": "Work closely with research scientists on state-of-the-art Natural Language Processing models, prompt engineering methodologies, and fine-tuning techniques.",
        "category": "Internship",
        "skills": "Python, PyTorch, Transformers, NLP",
        "location": "New York, NY / In-person",
        "eligibility": "Graduate students or advanced undergraduates with ML project experience",
        "deadline": "2025-03-30 (Demonstration Only)",
        "url": "https://example.com/ml-research-intern"
    },
    {
        "id": "OPP_016",
        "title": "Undergraduate Research Fellowship in AI Ethics",
        "description": "A paid semester-long fellowship targeting the ethical impacts, bias mitigation, and socio-economic consequences of automated artificial intelligence systems.",
        "category": "Fellowship",
        "skills": "Research Writing, Policy Analysis, AI Ethics",
        "location": "Remote",
        "eligibility": "Undergraduates in Humanities, Social Sciences, or STEM",
        "deadline": "2025-04-20 (Demonstration Only)",
        "url": "https://example.com/ethics-fellowship"
    },
    {
        "id": "OPP_017",
        "title": "Cloud Native Architect Bootcamp",
        "description": "An advanced technical training cohort specializing in distributed architectures, microservices scalability, and service meshes.",
        "category": "Bootcamp",
        "skills": "Kubernetes, Istio, Prometheus, Go",
        "location": "Chicago, IL / Hybrid",
        "eligibility": "Intermediate developers or IT professionals",
        "deadline": "2025-07-15 (Demonstration Only)",
        "url": "https://example.com/cloud-architect-bootcamp"
    },
    {
        "id": "OPP_018",
        "title": "DevOps Engineering Internship",
        "description": "A fast-paced summer program implementing server automation, managing deployment infrastructure, and scaling CI/CD scripts using GitHub Actions.",
        "category": "Internship",
        "skills": "Git, Jenkins, Docker, Linux administration, Bash",
        "location": "Remote",
        "eligibility": "Any active degree-seeking computer science student",
        "deadline": "2025-03-12 (Demonstration Only)",
        "url": "https://example.com/devops-intern"
    },
    {
        "id": "OPP_019",
        "title": "Cybersecurity Threat Hunting Workshop",
        "description": "Interactive technical seminar providing live labs for detecting, monitoring, and mitigating security breaches and active system network intrusions.",
        "category": "Workshop",
        "skills": "Wireshark, Splunk, Incident Response",
        "location": "Virtual",
        "eligibility": "Basic networking knowledge recommended",
        "deadline": "2025-04-05 (Demonstration Only)",
        "url": "https://example.com/threat-hunting"
    },
    {
        "id": "OPP_020",
        "title": "Next-Gen Fintech Hackathon",
        "description": "Collaborate with teams of developers to build innovative decentralized finance applications, secure payment gateways, or blockchain-based asset solutions.",
        "category": "Hackathon",
        "skills": "Solidity, Web3.js, React, Node.js",
        "location": "San Francisco, CA / Hybrid",
        "eligibility": "Open to all software developers and tech students",
        "deadline": "2025-05-25 (Demonstration Only)",
        "url": "https://example.com/fintech-hackathon"
    }
]

os.makedirs('data', exist_ok=True)
with open('data/opportunities.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=opportunities[0].keys())
    writer.writeheader()
    writer.writerows(opportunities)

print("Created self-contained dataset at 'data/opportunities.csv'!")

Created self-contained dataset at 'data/opportunities.csv'!


## 4. Clean and Validate the Dataset
We will load our dataset into a Pandas DataFrame, clean text columns, handle nulls or missing fields, and present a structural overview.

In [26]:
df = pd.read_csv('data/opportunities.csv')

# Handle missing values and string stripping
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna("").str.strip()

print(f"Dataset successfully loaded. Total Records: {len(df)}")
display(df.info())
display(df.head(2))

Dataset successfully loaded. Total Records: 20
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id           20 non-null     object
 1   title        20 non-null     object
 2   description  20 non-null     object
 3   category     20 non-null     object
 4   skills       20 non-null     object
 5   location     20 non-null     object
 6   eligibility  20 non-null     object
 7   deadline     20 non-null     object
 8   url          20 non-null     object
dtypes: object(9)
memory usage: 1.5+ KB


None

,id,title,description,category,skills,location,eligibility,deadline,url
0,OPP_001,Global AI Hackathon,A weekend hackathon focused on building genera...,Hackathon,"Python, PyTorch, Generative AI, Machine Learning",Virtual,All university students globally,2025-05-15 (Demonstration Only),https://example.com/global-ai-hackathon
1,OPP_002,Data Science Internship - NextGen Analytics,Summer internship program for aspiring data sc...,Internship,"SQL, Python, Pandas, Tableau","Chicago, IL / Hybrid",Rising juniors and seniors majoring in STEM fi...,2025-04-01 (Demonstration Only),https://example.com/nextgen-internship


## 5. Metadata Processing & Text Synthesis
Instead of complicated text chunking which could sever structural information for metadata-rich fields, we will combine our textual fields into one cohesive searchable document per opportunity.

In [27]:
def create_searchable_text(row):
    return f"Title: {row['title']}. Category: {row['category']}. Description: {row['description']}. Required Skills: {row['skills']}. Location: {row['location']}. Eligibility: {row['eligibility']}."

df['search_document'] = df.apply(create_searchable_text, axis=1)

# Verification check
print("Sample unified document index 0:\n")
print(df['search_document'].iloc[0])

Sample unified document index 0:

Title: Global AI Hackathon. Category: Hackathon. Description: A weekend hackathon focused on building generative AI applications and machine learning models to solve real-world climate issues.. Required Skills: Python, PyTorch, Generative AI, Machine Learning. Location: Virtual. Eligibility: All university students globally.


## 6. Pretrained Semantic Embedding Generation
We will download the lightweight `all-MiniLM-L6-v2` model from SentenceTransformers, map our searchable documents into high-dimensional vectors, and verify the model output structure.

In [28]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Test a trial embedding generation
test_vector = embedding_model.encode("Test search opportunity text")
print(f"Embedding dimension: {test_vector.shape[0]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding dimension: 384


## 7. ChromaDB Storage and Indexing
We will initialize an Ephemeral Client (in-memory) of ChromaDB. Next, we will import our data points containing distinct document strings, native metadata dictionary mappings, and corresponding identifier tags.

In [29]:
# Setup in-memory client
chroma_client = chromadb.Client()

# Create or get the collection
collection_name = "student_opportunities"
try:
    chroma_client.delete_collection(name=collection_name)
except Exception:
    pass

collection = chroma_client.create_collection(name=collection_name)

# Prepare dataset properties
documents = df['search_document'].tolist()
ids = df['id'].tolist()

# Format metadata for ChromaDB querying/filtering
metadata_list = df[['title', 'category', 'skills', 'location', 'eligibility', 'deadline', 'url']].to_dict(orient='records')

# Generate model embeddings
embeddings = embedding_model.encode(documents).tolist()

# Add elements directly into collection
collection.add(
    ids=ids,
    embeddings=embeddings,
    documents=documents,
    metadatas=metadata_list
)

print(f"Stored {collection.count()} elements inside the Vector Store!")

Stored 20 elements inside the Vector Store!


## 8. Implementation of the Reusable Semantic Search Engine

We will build a versatile `semantic_search` function.

### Distance vs. Similarity Explanation:
ChromaDB returns a distance metric (e.g., L2 squared distance). To present this as an intuitive similarity score to users:
- **L2 Distance:** A value of 0 indicates complete identity; higher numbers mean greater distance (lower similarity).
- **Cosine Similarity:** Typically ranges from 0 to 1, where 1 means identical. For cosine distance, we calculate similarity as `1.0 - distance`.

In [33]:
def semantic_search(query: str, n_results: int = 3, category_filter: str = None):
    if not query.strip():
        print("Empty search query received.")
        return []

    # Generate query embedding vector
    query_embedding = embedding_model.encode(query).tolist()

    # Setup categorical filters if provided
    where_clause = {}
    if category_filter:
        where_clause = {"category": category_filter}

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        where=where_clause
    )

    formatted_results = []
    if not results or not results['ids'] or len(results['ids'][0]) == 0:
        return formatted_results

    for idx in range(len(results['ids'][0])):
        distance = results['distances'][0][idx]

        # Standardize an L2 distance metric to an approximate percentage similarity score representation
        similarity_score = max(0.0, min(1.0, 1.0 - (distance / 2.0)))

        item = {
            "id": results['ids'][0][idx],
            "document": results['documents'][0][idx],
            "metadata": results['metadatas'][0][idx],
            "distance": distance,
            "similarity": similarity_score
        }
        formatted_results.append(item)

    return formatted_results

## 9. Demonstrating Search Queries
We will execute the 5 distinct queries requested, showcasing semantic matching without requiring exact keyword overlaps.

In [31]:
test_queries = [
    "I want to build AI projects and learn machine learning.",
    "Find competitions where software engineering students can work in teams.",
    "I need a scholarship or funded training opportunity.",
    "I want cybersecurity training for beginners.",
    "Find internships where I can improve my web development skills."
]

for i, q in enumerate(test_queries, 1):
    print(f"\n{'='*70}\nQuery #{i}: \"{q}\"\n{'='*70}")
    matches = semantic_search(query=q, n_results=2)

    for m in matches:
        meta = m['metadata']
        print(f"\n[Match ID: {m['id']}] Category: {meta['category']} | Est. Similarity: {m['similarity']:.2%}")
        print(f"Title: {meta['title']}")
        print(f"Location: {meta['location']} | Deadline: {meta['deadline']}")
        print(f"Skills required: {meta['skills']}")
        print(f"URL Reference: {meta['url']}")


Query #1: "I want to build AI projects and learn machine learning."

[Match ID: OPP_015] Category: Internship | Est. Similarity: 53.88%
Title: Machine Learning Research Internship
Location: New York, NY / In-person | Deadline: 2025-03-30 (Demonstration Only)
Skills required: Python, PyTorch, Transformers, NLP
URL Reference: https://example.com/ml-research-intern

[Match ID: OPP_009] Category: Fellowship | Est. Similarity: 50.10%
Title: AI Product Management Fellowship
Location: Remote | Deadline: 2025-04-10 (Demonstration Only)
Skills required: Product Strategy, Product Design, AI/ML concepts
URL Reference: https://example.com/pm-fellowship

Query #2: "Find competitions where software engineering students can work in teams."

[Match ID: OPP_018] Category: Internship | Est. Similarity: 48.27%
Title: DevOps Engineering Internship
Location: Remote | Deadline: 2025-03-12 (Demonstration Only)
Skills required: Git, Jenkins, Docker, Linux administration, Bash
URL Reference: https://example.c

## 10. Advanced Search with Metadata Filtering
Let's test search using the `category_filter` parameter to see metadata filtering in action.

In [32]:
filtered_query = "I want to build software on AWS"
print(f"Query: '{filtered_query}' (Category Filter: 'Workshop')")
results = semantic_search(query=filtered_query, n_results=2, category_filter="Workshop")

for r in results:
    meta = r['metadata']
    print(f"-> Title: {meta['title']} | Category: {meta['category']} | Sim: {r['similarity']:.2%}")

Query: 'I want to build software on AWS' (Category Filter: 'Workshop')
-> Title: Cloud Computing and AWS Workshop | Category: Workshop | Sim: 52.98%
-> Title: Git and GitHub Collaboration Seminar | Category: Workshop | Sim: 23.93%


## 11. Project Evaluation & Summary

### Retrieval Assessment
- The embeddings successfully mapped semantic concepts (e.g., matching the query "cybersecurity training" to a "CTF Competition" or a "Threat Hunting Workshop" even when text descriptions did not match perfectly).
- Combining fields into cohesive structural search documents ensures that skills and category tags are indexed along with descriptions.
- Categorical filters function natively inside ChromaDB query clauses, minimizing irrelevant results when searching for specific formats.

### Limitations & Future Advancements
1. **Fixed Embeddings:** Pretrained transformer vectors might struggle with specialized custom developer shorthand vocabulary without fine-tuning.
2. **Threshold Sensitivity:** Relying on basic L2 thresholds can sometimes filter out relevant items since distance metrics scale based on text length and dimension counts. Future updates could incorporate cross-encoder re-ranking models for better precision.